# Mulligan Analysis
The goal is to analyze different mulligan rules from various perspectives using hypergeometric distributions, which will be familiar to those working with card games.

## Inital thoughts

### The context
The context of this is a game called Hubworld: Aidalon. The goal in this game is to capture an opponents 'agents' before they capture your agents. As this is the win condition, it is often preferable to not have agents in your opening hand. It is also the case that other cards are necessary for defending as well as for generating income. Having these in an opening hand is ideal. Other strategies, 'mulliganing hard' for core pieces of your deck are also viable.

Our goal is not to determine what the best strategy is but to determine which mulligans best support certain outcomes. We will try to be agnostic and explain the relevant mathematics when it arises.

### Expected Value and probability
The most familiar way of measuring an undesireable outcome is to state the probability of it ovvuring, or of it not occuring. However, there may be another more relevant probabilistic measure: Expected Value. In this case, we could ask what the expected value is of the number of agents in an opening hand of 5 cards is, without discarding any cards. This would be computed as follows. The expected value of the number of agents would be computed as:

$$E[\text{Number of Agents}]= \Sigma_{k=0}^5 P(\text{k Agents})\cdot k$$

It is a weighted sum that can be very useful. We will compute this when comparing different mulligan systems as well as the standard probabilities.

### Spectrum of Strategies
Ideally, we would like to compare different approaches to a mulligan, but this is currently a bit unwieldy. For the time being we will look at minimizing the number of agents as well as maximizing the number of agents. This will give us some bounds on what the situatoin looks like.

### General approach
We will break up the calculation by considering what happens in the different cases that we draw $k$ agents. We will condition our expected value and probability computations on this and then aggregate the values.

### Relevant constants
In the game in question, there are $36$ cards to a deck and $6$ of them are agents. There are other types of cards, economic, defensive, and offensive cards. The rest of the composition of the deck is up to the player (aside from adherence to rules that do not impact the distribution of such cards).

We will use $t$ to denote the total number of cards in the deck, $n$ to denote the number of agents in the deck (marked cards), $h_0$ to denote number of cards drawn initially, $h_1$ to denote the size of the final hand, and the variables $k$ and $m$ will be reserved for the number of agents in the initial draw and final hand, respectively.

Perhaps it would be more unifying to have $k$ be the number of cards discarded. Really, those should be separate variables though.

You can change the values below, execute the cell, and then rerun all subsequent cells to see what impact these decisions have on the various metrics. Default values are given in comment after each assignment.

In [2]:
t = 36 # 36 total cards in the deck
n = 6 # 6 agents in the deck
h_0 = 5 # 5 cards in the initial hand
h_1 = 5 # 5 cards in the hand after mulligan

We will stick to using the default values when explaining things as it will be more relevant that way for the intended audience.


## The 'Friendliest Mulligan'
This is the current mulligan system. It is the one found in Earthborne Rangers (and in Arkham Horror the Card Game, albeit modified). The player draws an initial hand of 5 cards. They set aside any number of cards from this initial draw and then redraw to a hand size of 5 cards. The set aside cards are then shuffled back into the deck. The downside of this is that it is possible to set aside cards and then draw into agents. This makes a good computation difficult as there may be an exploding tree of situations to condition your computation on. As mentioned, we will, for the time being, only consider the strategy of minimizing the number of agents or maximizing them.

### The Initial Draw
We will now do some baseline computations for the initial draw to see what is expected before you discard cards from your opening hand.

#### Baseline probabilities
Letes consider the initial draw. The probability of drawing $k$ agents from a standard deck among k cards is given by the formula

$$P(\text{k Agents})=\frac{\binom{6}{k}\binom{31}{5-k}}{\binom{36}{5}}$$

or abstractly

$$P(\text{k Agents})=\frac{\binom{n}{k}\binom{t-n}{h_0-k}}{\binom{t}{h_0}}.$$

We can use this to compute the probabilities of getting $k$ agents in our initial draw.

In [ ]:
import numpy as np
import pandas as pd
from scipy.special import comb

# Calculate probabilities for k agents (k = 0 to 5)
# P(k Agents) = C(6,k) * C(30,5-k) / C(36,5)

k_values = np.arange(0, min(n, h_0) + 1)  # k can only be from 0 to min(n, h_0)
probabilities = {}

for k in k_values:
    num = comb(n, k, exact=True) * comb(t - n, h_0 - k, exact=True)
    denom = comb(t, h_0, exact=True)
    prob = num / denom
    probabilities[k] = prob

# Create DataFrame
df = pd.DataFrame({
    'k (Agents)': k_values,
    'Probability': [f"{p*100:.2f}%" for p in probabilities.values()]
})

#print(df.to_string(index=False))
df

,k (Agents),Probability
0,0,37.80%
1,1,43.62%
2,2,16.15%
3,3,2.31%
4,4,0.12%
5,5,0.00%


#### Expected Number of Agents in the Initial Draw

This can then be used to compute the expected number of agents in the initial draw:
$$E[\text{Number of Agents}]=\Sigma_{k=0}^{5}P(\text{k Agents}) \cdot 
$$

In [25]:
expected_value = 0
for k in range(0, min(n, h_0) + 1):
    contribution = probabilities[k] * k
    expected_value += contribution
    print(f"k = {k}: P({k} Agents) × {k} = {probabilities[k]:.8f} × {k} = {contribution:.6f}")


print(f"Expected Value = {expected_value:.3f} agents")

k = 0: P(0 Agents) × 0 = 0.37800802 × 0 = 0.000000
k = 1: P(1 Agents) × 1 = 0.43616310 × 1 = 0.436163
k = 2: P(2 Agents) × 2 = 0.16154189 × 2 = 0.323084
k = 3: P(3 Agents) × 3 = 0.02307741 × 3 = 0.069232
k = 4: P(4 Agents) × 4 = 0.00119366 × 4 = 0.004775
k = 5: P(5 Agents) × 5 = 0.00001592 × 5 = 0.000080
Expected Value = 0.833 agents


Thus the expected number of agents is  $0.833=\frac{5}{6}$. This means that in the initial draw you are expecting to see 1 agent.

### After the Mulligan
We will now look at what the distribution looks like at the end of the mulligan process. We will do this by breaking up the situation into different cases and computing conditional probabilities and expected values. This follows Adams Law, which states:

$$ E[X] = E[E[X|Y]]. $$

This is precisely what allows us to breakdown a complicated situation into many different subcases. In the context of marginal distributions, the analogue of Adams Law is called the law of total probabilities:

$$ P(A) = \Sigma_i P(A | B_i) \cdot P(B_i) $$

where the $B_i$ partition the sample space. We will be partitioning the sample space based on the number of agents in the initial draw.

#### $P(\text{m Agents} | \text{k Agents})$
We will compute $P(\text{m Agents} | \text{k Agents})$ for various $m$ and $k$ where $m4 is the number of agents in the final hand after the mulligan and $k$ is the number of agents in the initial draw. Again, this is assuming the strategy is to discard a card if and only if it is an agents.
Recall that the definition of conditional probability leads to

$$ P(\text{m Agents}|\text{k Agents})= \frac{P(\text{m Agents}\cap\text{k Agents})}{P(\text{k Agents})}.$$

Abstractly, this ends up being

$$ P(\text{m Agents} | \text{k Agents})= \frac{\binom{6-k}{m} \cdot \binom{31-(6-k)}{k-m}}{\binom{31}{k}}.$$

This can be explained as follows. You draw $5$ cards in the initial draw, $k$ of them are agents. There are now $31$ remaining cards in the deck and $6-k$ of them are agents. So there are $\binom{31-(6-k)}{k-m}$ ways of choosing non-agents and $\binom{6-k}{m}$ ways of choosing agents when redrawing $k$ cards out of a total of $\binom{31}{k}$ total ways of redrawing $k$ cards.

In [14]:
from scipy.special import comb

# Calculate P(m | k) for all k and m from 0 to 5
# Formula: P(m Agents | k Agents) = C(6-k, m) * C(25+k, k-m) / C(31, k)
# This represents redrawing k cards from 31, where 6-k are agents and 25+k are non-agents

conditional_probs = {}
calculation_details = {}

for k in range(6):
    conditional_probs[k] = {}
    calculation_details[k] = {}
    
    # Get P(k agents) from previous calculation
    p_k = probabilities[k]
    
    for m in range(6):
        # Calculate intermediate values
        num_a = 0 if m > 6 - k else comb(6 - k, m, exact=True)
        num_b = 0 if k - m > 25 + k else comb(25 + k, k - m, exact=True)
        num = 0 if m > k else num_a * num_b
        denom = comb(31, k, exact=True)
        
        if denom > 0:
            prob_conditional = num / denom
        else:
            prob_conditional = 0.0
        
        # Joint probability P(m and k) = P(m | k) * P(k)
        prob_joint = prob_conditional * p_k
        
        conditional_probs[k][m] = prob_conditional
        calculation_details[k][m] = {
            'num_a': num_a,
            'num_b': num_b,
            'num': num,
            'denom': denom,
            'p_conditional': prob_conditional,
            'p_joint': prob_joint,
            'p_k': p_k
        }

# Create a comprehensive DataFrame with all pairs (k, m) and all calculation details
data = []
for k in range(6):
    for m in range(6):
        details = calculation_details[k][m]
        data.append({
            'k': k,
            'm': m,
            'num_a': details['num_a'],
            'num_b': details['num_b'],
            'num': details['num'],
            'denom': details['denom'],
            'P(m|k)': details['p_conditional'],
            'P(k)': details['p_k'],
            'P(m and k)': details['p_joint'],
            'P(m|k) %': f"{details['p_conditional']*100:.2f}%",
            'P(k) %': f"{details['p_k']*100:.2f}%",
            'P(m and k) %': f"{details['p_joint']*100:.2f}%"
        })

df_all = pd.DataFrame(data)


We now want to display the results. For reasons of space we will omit all rows where the value of the num is 0 (and so the resulting probability will be 0 as well).

In [15]:

df_display = df_all[df_all['num'] != 0]
print("Full Conditional Probability Table with Calculation Details")
print("=" * 150)
print("Columns in dataframe: k (initial agents), m (final agents), num_a, num_b, num, denom, P(m|k), P(k), P(m and k), P(m|k) %, P(k) %, P(m and k) %")
print("=" * 150)
print(df_display.to_string(index=False))
print("=" * 150)


Full Conditional Probability Table with Calculation Details
Columns in dataframe: k (initial agents), m (final agents), num_a, num_b, num, denom, P(m|k), P(k), P(m and k), P(m|k) %, P(k) %, P(m and k) %
 k  m  num_a  num_b    num  denom   P(m|k)     P(k)  P(m and k) P(m|k) % P(k) % P(m and k) %
 0  0      1      1      1      1 1.000000 0.378008    0.378008  100.00% 37.80%       37.80%
 1  0      1     26     26     31 0.838710 0.436163    0.365814   83.87% 43.62%       36.58%
 1  1      5      1      5     31 0.161290 0.436163    0.070349   16.13% 43.62%        7.03%
 2  0      1    351    351    465 0.754839 0.161542    0.121938   75.48% 16.15%       12.19%
 2  1      4     27    108    465 0.232258 0.161542    0.037519   23.23% 16.15%        3.75%
 2  2      6      1      6    465 0.012903 0.161542    0.002084    1.29% 16.15%        0.21%
 3  0      1   3276   3276   4495 0.728810 0.023077    0.016819   72.88%  2.31%        1.68%
 3  1      3    378   1134   4495 0.252280 0.023077  

To zoom in on particular cases, enter values in the list for the number of final agents you want to consider and then execute the following cells. If the situation is impossible "N/A (invalid)" will be listed. If $k$ is zero then "redraw" will be listed.

In [21]:
possible_m_values = [0,1,2]
print(f"Calculations for m = {m} agents in final hand:")

Calculations for m = 3 agents in final hand:


In [24]:

# Also display tables for each fixed m
print("\n\nDetailed View by Fixed m (Final Agent Count):")
print("=" * 80)

for m in possible_m_values:
    print(f"\nFor m = {m} agents (final hand):")
    print("-" * 80)
    
    data_m = []
    for k in range(6):
        details = calculation_details[k][m]
        
        if k == 0:
            calc_str = "No redraw"
        elif details['num'] == 0:
            calc_str = "N/A (invalid)"
        else:
            calc_str = f"C({6-k},{m}) × C({25+k},{k-m}) / C(31,{k})"
        
        data_m.append({
            'k (Initial Agents)': k,
            'Calculation': calc_str,
            'P(m|k)': f"{details['p_conditional']:.6f}",
            'P(k)': f"{details['p_k']:.6f}",
            'P(m and k)': f"{details['p_joint']:.6f}",
            'Percentage': f"{details['p_conditional']*100:.2f}%"
        })
    
    df_m = pd.DataFrame(data_m)
    print(df_m.to_string(index=False))

print("\n" + "=" * 80)



Detailed View by Fixed m (Final Agent Count):

For m = 0 agents (final hand):
--------------------------------------------------------------------------------
 k (Initial Agents)                Calculation   P(m|k)     P(k) P(m and k) Percentage
                  0                  No redraw 1.000000 0.378008   0.378008    100.00%
                  1 C(5,0) × C(26,1) / C(31,1) 0.838710 0.436163   0.365814     83.87%
                  2 C(4,0) × C(27,2) / C(31,2) 0.754839 0.161542   0.121938     75.48%
                  3 C(3,0) × C(28,3) / C(31,3) 0.728810 0.023077   0.016819     72.88%
                  4 C(2,0) × C(29,4) / C(31,4) 0.754839 0.001194   0.000901     75.48%
                  5 C(1,0) × C(30,5) / C(31,5) 0.838710 0.000016   0.000013     83.87%

For m = 1 agents (final hand):
--------------------------------------------------------------------------------
 k (Initial Agents)                Calculation   P(m|k)     P(k) P(m and k) Percentage
                  0           

Now we are in a position to compute the probability of having $m$ agents after the mulligan. We will use that conditioning on $k$ makes each scenario distinct and so we are able to use additivity of disjoint events. The inclusion-exclusion principle says that 

$$ P(A \cup B) = P(A) + P(B) - P(A \cap B).$$

In english, this reads as "the probability of A or B happening is the probability of A plus the probability of B minus the probability of A and B." This follows from the fact that if A and B occur then that instance is in effect double counted, and so we subtract all of these double counted instances. However, if A and B can not both happen then we say they are disjoint and arrive at

$$ P(A \cup B) = P(A) + P(B).$$

In practice, our $A$ will be $m$ agents in the final hand and $k_0$ agents in the initial draw while $B$ will be $m$ agents in the final hand and $k_1$ agents in the initial draw for different values of $k_0$ and $k_1$. Obviously, since $k_0\neq k_1$, these events are disjoint and $P(A \cap B)=0.$ 
Using these disjoint cases, we arrive at 

$$ P(\text{m Agents}) = \Sigma_{k=0}^{5} P(\text{m Agents}\cap \text{k Agents}).$$

But we have computed these values above via

$$P(\text{m Agents} \cap\text{k Agents}) = P(\text{m Agents}|\text{k Agents})\cdot P(\text{k Agents}),$$

which is essentially the definition.
We thus get the following table of values. Some of the percentages are miniscule. To see more than the first decimal place, change `1` in the `format_percentage` function to the number of decimal places you would like to see.

In [ ]:
percentage_decimals = 1
def format_percentage(x):
    return f"{x*100:.{percentage_decimals}f}%"

In [36]:
marg_data = []

for m in range(6):
    row = {'m': m}
    
    # Add P(m|k)*P(k) for each k (0 to 5) as numeric values first
    total_prob = 0
    for k in range(6):
        p_m_given_k = calculation_details[k][m]['p_conditional']
        p_k = probabilities[k]
        row[f'P(m & {k})'] = p_m_given_k * p_k
        total_prob += p_m_given_k * p_k
    
    # Final column: marginal probability P(m)
    row['P(m)'] = total_prob
    
    marg_data.append(row)

df_marginal = pd.DataFrame(marg_data)

# Format the probability columns as percentages with 1 decimal place
prob_columns = [col for col in df_marginal.columns if col != 'm']
df_marginal_formatted = df_marginal.copy()
for col in prob_columns:
    df_marginal_formatted[col] = df_marginal_formatted[col].apply(format_percentage)

print("Marginal Probabilities: P(m agents in final hand)")
print("=" * 140)
print("Each row is a value of m, columns show P(m and k) for k=0 to 5, final column is P(m)")
print("=" * 140)
print(df_marginal_formatted.to_string(index=False))
print("=" * 140)


Marginal Probabilities: P(m agents in final hand)
Each row is a value of m, columns show P(m and k) for k=0 to 5, final column is P(m)
 m P(m & 0) P(m & 1) P(m & 2) P(m & 3) P(m & 4) P(m & 5)  P(m)
 0    37.8%    36.6%    12.2%     1.7%     0.1%     0.0% 88.3%
 1     0.0%     7.0%     3.8%     0.6%     0.0%     0.0% 11.4%
 2     0.0%     0.0%     0.2%     0.0%     0.0%     0.0%  0.3%
 3     0.0%     0.0%     0.0%     0.0%     0.0%     0.0%  0.0%
 4     0.0%     0.0%     0.0%     0.0%     0.0%     0.0%  0.0%
 5     0.0%     0.0%     0.0%     0.0%     0.0%     0.0%  0.0%


#### $E[\text{? Agents}|\text{k Agents}]$
We now turn to the computation of expected value. We approach the final expected value using Adams law, so we will need these conditional expected value computations first. By definition, we have that 

$$E[X|Y=y]=\Sigma_x x\cdot P(X=x|Y=y)$$

where $x$ is the possible values of the random variable $X$. In our case of interest, we would get

$$E[\text{final agent count}|\text{k Agents}]=\Sigma_{m=0}^{m=5} m\cdot P(\text{m Agents}|\text{k Agents}).$$

In [44]:
decimals=3
def set_decimal_places(x):
    return f"{x:.{decimals}f}"

In [48]:
# Create a dataframe for conditional expected values
# E[final agents | k agents] = Σ(m=0 to 5) m * P(m|k)

# First, build a dataframe with one row per k, showing P(m|k) for each m
ce_data = []

for k in range(6):
    row = {'k': k}
    
    # Add P(m|k) for each m
    for m in range(6):
        p_m_given_k = calculation_details[k][m]['p_conditional']
        row[f'P({m}|k)'] = p_m_given_k
    
    ce_data.append(row)

df_conditional = pd.DataFrame(ce_data)

# Now add columns for m * P(m|k) and compute conditional expected value
for m in range(6):
    col_name = f'P({m}|k)'
    df_conditional[f'{m}*P({m}|k)'] = m * df_conditional[col_name]

# Sum across for each k to get E[final agents | k]
df_conditional['E[final|k]'] = df_conditional[[f'{m}*P({m}|k)' for m in range(6)]].sum(axis=1)

df_conditional_display = df_conditional.copy()
df_conditional_display=df_conditional_display[['k'] + [f'{m}*P({m}|k)' for m in range(6)] + ['E[final|k]']]

disp_columns = [col for col in df_conditional_display.columns if col != 'k']
for col in disp_columns:
    df_conditional_display[col] = df_conditional_display[col].apply(set_decimal_places)



print("Conditional Expected Values: E[final agents | k initial agents]")
print("=" * 100)
print(df_conditional_display.to_string(index=False))
print("=" * 100)


Conditional Expected Values: E[final agents | k initial agents]
 k 0*P(0|k) 1*P(1|k) 2*P(2|k) 3*P(3|k) 4*P(4|k) 5*P(5|k) E[final|k]
 0    0.000    0.000    0.000    0.000    0.000    0.000      0.000
 1    0.000    0.161    0.000    0.000    0.000    0.000      0.161
 2    0.000    0.232    0.026    0.000    0.000    0.000      0.258
 3    0.000    0.252    0.037    0.001    0.000    0.000      0.290
 4    0.000    0.232    0.026    0.000    0.000    0.000      0.258
 5    0.000    0.161    0.000    0.000    0.000    0.000      0.161


We thus arrive at our final number of expected agents after the mulligan:

In [47]:

print(f"E[final agents] = {df_conditional['E[final|k]'].sum():.3f} agents")


E[final agents] = 1.129 agents


### Draw 7 Keep 5

This mulligan system is simpler than the Friendliest Mulligan: the player draws 7 cards and then must discard exactly 2 cards. There is no redrawing—the discarded cards are simply shuffled back into the deck. Again, assuming the strategy of minimizing the number of agents, the player will discard agents first, and only discard non-agents if there are not enough agents to discard.

#### The Initial Draw

The probability of drawing $k$ agents in an initial draw of 7 cards is given by:

$$P(\text{k Agents})=\frac{\binom{n}{k}\binom{t-n}{h_0-k}}{\binom{t}{h_0}}$$

With our default values, this becomes:

$$P(\text{k Agents})=\frac{\binom{6}{k}\binom{30}{7-k}}{\binom{36}{7}}.$$


After drawing 7 cards with $k$ agents, you must discard exactly 2 cards. Using the minimize strategy, you discard agents first. This means:

- If $k \geq 2$: You discard 2 agents, leaving $k - 2$ agents in your final hand
- If $k = 1$: You discard the 1 agent and 1 non-agent, leaving 0 agents
- If $k = 0$: You discard 2 non-agents, leaving 0 agents

In general, the number of agents in your final hand is $m = \max(0, k - 2)$, which is a deterministic function of $k$. Therefore, $P(\text{m Agents}|\text{k Agents})$ is either 0 or 1 and so we will not need conditional probabilities.

In [52]:
# Update parameters for Draw 7, Keep 5
h_0_draw7 = 7

# Set up percentage formatting function
percentage_decimals = 4
def format_percentage(x):
    return f"{x*100:.{percentage_decimals}f}%"

# Recalculate probabilities for initial draw of 7
k_values_7 = np.arange(0, min(n, h_0_draw7) + 1)
probabilities_7 = {}

for k in k_values_7:
    num = comb(n, k, exact=True) * comb(t - n, h_0_draw7 - k, exact=True)
    denom = comb(t, h_0_draw7, exact=True)
    prob = num / denom
    probabilities_7[k] = prob

# Display the initial probabilities
df_7_initial = pd.DataFrame({
    'k (Agents)': k_values_7,
    'Probability': [format_percentage(p) for p in probabilities_7.values()]
})

print("Initial Draw Probabilities (Draw 7)")
print("=" * 80)
print(df_7_initial.to_string(index=False))
print("=" * 80)

# Calculate expected value for initial draw
expected_value_7 = sum(probabilities_7[k] * k for k in k_values_7)
print(f"\nExpected value in initial draw = {expected_value_7:.3f} agents")
print("=" * 80)


Initial Draw Probabilities (Draw 7)
 k (Agents) Probability
          0    24.3876%
          1    42.6783%
          2    25.6070%
          3     6.5659%
          4     0.7295%
          5     0.0313%
          6     0.0004%

Expected value in initial draw = 1.167 agents


#### $P(\text{m Agents})$ - Draw 7 Keep 5

The process of discarding $2$ cards and not redrawing means that some of the probabilities aggregate as follows.

- $P(\text{0 Agents in final hand})=P(\text{initial Agents} \leq2).$ Thus we have that $P(m=0)=P(k=0)+P(k=1)+P(k=2).$
- For $0<j<5$ we have that $P(m=j)=P(k=j+2).$
- The max number of possible agents in a hand is $4$ and this can only happen if you draw all $6$ agents in the intial $7$ cards.


In [55]:
# Final probabilities after discarding 2 cards
# Create a table showing k → m transformation and final probabilities

print("\nFinal Agent Probabilities - Draw 7 Keep 5")
print("=" * 100)
print("After drawing 7 cards and discarding 2, the transformation is: m = max(0, k - 2)")
print("=" * 100)

# Build transformation table
transform_data = []
prob_by_final = {}

for k in k_values_7:
    m = max(0, k - 2)
    p_k = probabilities_7[k]
    
    transform_data.append({
        'k (Drawn)': k,
        'm (Final)': m,
        'P(k)': p_k,
        'P(k) %': format_percentage(p_k)
    })
    
    # Accumulate probabilities for each final m
    if m not in prob_by_final:
        prob_by_final[m] = 0.0
    prob_by_final[m] += p_k

df_transform = pd.DataFrame(transform_data)
print("\nTransformation: Initial Agents → Final Agents")
print("-" * 100)
print(df_transform[['k (Drawn)', 'm (Final)', 'P(k) %']].to_string(index=False))

print("\n" + "=" * 100)
print("Final Distribution: Probability of m agents in final hand")
print("=" * 100)

# Create final distribution table
final_dist_data = []
for m in sorted(prob_by_final.keys()):
    final_dist_data.append({
        'm (Final)': m,
        'P(m)': prob_by_final[m],
        'P(m) %': format_percentage(prob_by_final[m])
    })

df_final = pd.DataFrame(final_dist_data)
print(df_final[['m (Final)', 'P(m) %']].to_string(index=False))
print("=" * 100)



Final Agent Probabilities - Draw 7 Keep 5
After drawing 7 cards and discarding 2, the transformation is: m = max(0, k - 2)

Transformation: Initial Agents → Final Agents
----------------------------------------------------------------------------------------------------
 k (Drawn)  m (Final)   P(k) %
         0          0 24.3876%
         1          0 42.6783%
         2          0 25.6070%
         3          1  6.5659%
         4          2  0.7295%
         5          3  0.0313%
         6          4  0.0004%

Final Distribution: Probability of m agents in final hand
 m (Final)   P(m) %
         0 92.6729%
         1  6.5659%
         2  0.7295%
         3  0.0313%
         4  0.0004%


#### $E[\text{? Agents}]$ - Draw 7 Keep 5

This can be computed directly following the definition. 

In [57]:

# Calculate expected value
expected_final = sum(m * prob_by_final[m] for m in prob_by_final.keys())
print(f"\nExpected number of agents in final hand: {expected_final:.3f}")


Expected number of agents in final hand: 0.081


## Comparison

We are now in a position to compare the two different methods.

In [59]:
# Create comparison table
print("\n" + "=" * 120)
print("COMPARISON: Final Agent Distribution P(m agents)")
print("=" * 120)

# Extract P(m) values for both strategies
comparison_data = []

for m in range(6):
    # Friendliest Mulligan: P(m) from df_marginal
    p_m_friendly = df_marginal[df_marginal['m'] == m]['P(m)'].values[0] if len(df_marginal[df_marginal['m'] == m]) > 0 else 0.0
    
    # Draw 7 Keep 5: P(m) from prob_by_final
    p_m_draw7 = prob_by_final.get(m, 0.0)
    
    comparison_data.append({
        'm (Agents)': m,
        'Friendliest %': format_percentage(p_m_friendly),
        'Draw 7 Keep 5 %': format_percentage(p_m_draw7),
        'Difference %': format_percentage(p_m_draw7 - p_m_friendly) if abs(p_m_draw7 - p_m_friendly) > 1e-10 else "—"
    })

df_comparison = pd.DataFrame(comparison_data)
print(df_comparison.to_string(index=False))
print("=" * 120)

print("\n" + "=" * 120)
print("COMPARISON: Expected Number of Agents in Final Hand")
print("=" * 120)

# Get expected values
ev_friendly = df_conditional['E[final|k]'].sum()
ev_draw7 = expected_final

print(f"{'Strategy':<25} {'Expected Agents':>20} {'Comparison':>30}")
print("-" * 120)
print(f"{'Friendliest Mulligan':<25} {ev_friendly:>20.3f}")
print(f"{'Draw 7 Keep 5':<25} {ev_draw7:>20.3f}")
print("-" * 120)
print(f"{'Difference (Draw 7 - Friendly)':<25} {ev_draw7 - ev_friendly:>+20.3f}")
print("=" * 120)

if ev_friendly < ev_draw7:
    print(f"\nFriendliest Mulligan is BETTER (reduces agents by {ev_friendly - ev_draw7:.3f})")
elif ev_draw7 < ev_friendly:
    print(f"\nDraw 7 Keep 5 is BETTER (reduces agents by {ev_draw7 - ev_friendly:.3f})")
else:
    print("\nBoth strategies have EQUAL effectiveness")


COMPARISON: Final Agent Distribution P(m agents)
 m (Agents) Friendliest % Draw 7 Keep 5 % Difference %
          0      88.3494%        92.6729%      4.3236%
          1      11.3970%         6.5659%     -4.8311%
          2       0.2531%         0.7295%      0.4764%
          3       0.0005%         0.0313%      0.0308%
          4       0.0000%         0.0004%      0.0004%
          5       0.0000%         0.0000%            —

COMPARISON: Expected Number of Agents in Final Hand
Strategy                       Expected Agents                     Comparison
------------------------------------------------------------------------------------------------------------------------
Friendliest Mulligan                     1.129
Draw 7 Keep 5                            0.081
------------------------------------------------------------------------------------------------------------------------
Difference (Draw 7 - Friendly)               -1.048

Draw 7 Keep 5 is BETTER (reduces agents by -1